In [2]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

#加载与预处理数据（与模块二完全一致）
DATA_PATH = "./data_tiny_shakespeare/input.txt"

#硬件与基础配置 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

#加载数据集
with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

dataset = DatasetDict({
    "train": Dataset.from_dict({"text": [text]})
})
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(examples):
    concatenated_examples = [k for k in examples["text"]]
    tokenized = tokenizer(
        concatenated_examples,
        truncation=True,
        max_length=512,
        return_overflowing_tokens=False,
    )
    return tokenized

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#加载全量微调模型
model = GPT2LMHeadModel.from_pretrained(
    "gpt2",
    #torch_dtype=torch.float16,
    #use_cache=False
).to(device)

model.gradient_checkpointing_enable()  # 必须开，否则8G显存绝对不够

#训练参数（全量微调的极致显存优化）
training_args = TrainingArguments(
    output_dir="./gpt2-full-finetune",
    learning_rate=5e-5,  # 全量微调学习率必须小
    per_device_train_batch_size=1,  # 批次只能1，梯度累积补
    gradient_accumulation_steps=8,  # 累积8步，等效batch=8
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    #fp16=True,
    logging_steps=1,
    report_to="none",
)

#开始训练
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)
trainer.train()

#生成测试
def generate_text(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

2026-03-28 09:32:16.535746: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-28 09:32:16.535883: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-28 09:32:16.582797: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


使用设备: cuda


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/hexiaoya/miniconda3/envs/ai/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss
1,0.535800
2,0.470300
3,0.448900


/home/hexiaoya/miniconda3/envs/ai/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
/home/hexiaoya/miniconda3/envs/ai/lib/python3.10/site-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


In [ ]:
TITLE="I am god"
print("提示词:"+TITLE)
print("续写内容:\n"+generate_text(TITLE))

提示词:I am god
